# State Objective Alignment Screen v1

"
        "This notebook consumes `state_objective_training_target_v1/training_targets.csv` and scores existing candidate daily paths against the state-conditioned training/checkpoint-selection target.

"
        "Governance:
"
        "- canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
"
        "- direct comparison baseline: `state_objective_training_target_v1`
"
        "- role: `diagnose`
"
        "- changed layer: `model_selection`, `reward`
"
        "- target: `>=30pp` relative return and `<=15%` MaxDD, with rebound participation and downtrend risk-cap target terms

"
        "External basis: Qlib artifact-first research workflow, FinRL-style separation of environment/reward/action behavior, and backtest-overfitting discipline requiring out-of-sample / walk-forward checks before promotion.
"
        

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / 'src' / 'soxl_quant').exists():
        REPO_ROOT = parent
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_training_targets import (
    score_state_objective_target_alignment,
    summarize_state_objective_target_alignment,
)

EXP = 'state_objective_alignment_screen_v1'
OUT_DIR = REPO_ROOT / 'runtime/model_selection/card' / EXP
TARGET_DIR = REPO_ROOT / 'runtime/model_selection/card/state_objective_training_target_v1'
SCREEN_DIR = REPO_ROOT / 'runtime/model_selection/card/state_conditioned_objective_screen_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

target_selection = json.loads((TARGET_DIR / 'selection.json').read_text())
screen_selection = json.loads((SCREEN_DIR / 'selection.json').read_text())
targets = pd.read_csv(TARGET_DIR / 'training_targets.csv')
manifest = pd.read_csv(SCREEN_DIR / 'candidate_path_manifest.csv')
manifest = manifest[manifest['exists'].astype(bool)].copy()
(len(targets), manifest['candidate'].nunique(), sorted(manifest['window'].unique()))


(2322, 8, ['strict_oos', 'wf_2020', 'wf_2022', 'wf_2023_2024'])

In [2]:
candidate_paths = {}
for row in manifest.itertuples(index=False):
    candidate_paths.setdefault(row.candidate, {})[row.window] = pd.read_csv(REPO_ROOT / row.path)

scored = score_state_objective_target_alignment(candidate_paths, targets)
summary = summarize_state_objective_target_alignment(scored)

scores_path = OUT_DIR / 'alignment_scores.csv'
summary_path = OUT_DIR / 'summary.json'
scored.to_csv(scores_path, index=False)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

summary


{'decision': 'target_alignment_no_promotion',
 'selected_candidate': 'state_conditioned_rebound_risk_router_v1',
 'selected_alignment_loss': 1.7085418563344201,
 'selected_window_gate': False,
 'selected_coverage_gate': True,
 'selected_missing_target_ratio': 0.0,
 'next_mechanism': 'train_or_checkpoint_score_state_objective_targets',
 'required_terms': ['target_position_alignment_loss',
  'rebound_participation_alignment_loss',
  'downtrend_risk_cap_violation_loss',
  'walkforward_gate_gap_loss']}

In [3]:
display(Markdown('## Alignment Ranking'))
display(scored)

display(Markdown('## Top Candidate vs Teacher Target Readout'))
top_candidate = summary['selected_candidate']
focus = scored[scored['candidate'] == top_candidate].iloc[0]
display(pd.DataFrame([focus]))


## Alignment Ranking

,candidate,target_rows,matched_rows,missing_target_ratio,coverage_gate,position_tracking_mae,rebound_position_mae,risk_cap_violation,window_gate_loss,max_relative_gap,max_drawdown_gap,window_gate,alignment_loss
0,state_conditioned_rebound_risk_router_v1,2322,2322,0.0,True,0.000000,0.151144,0.015691,134.348893,77.569598,2.939129,False,1.708542
1,recovery_reentry_conditional_vol_target_v1,2322,2322,0.0,True,0.253706,0.208537,0.270763,356.920108,0.000000,15.923275,False,5.323034
2,recovery_reentry_cppi_floor_v1,2322,2322,0.0,True,0.265752,0.261291,0.279327,404.543049,0.000000,17.120180,False,5.951073
3,current_coverage_residual_guard_refresh_v1,2322,2322,0.0,True,0.201421,0.207981,0.192123,483.891119,155.130239,19.266345,False,6.224786
4,recovery_reentry_equity_drawdown_guard_v1,2322,2322,0.0,True,0.194548,0.178789,0.189878,565.721828,45.413925,19.365251,False,6.968854
5,recovery_reentry_anchor_confirm_v1,2322,2322,0.0,True,0.273907,0.204303,0.310833,608.266964,0.000000,26.587207,False,8.008514
6,current_coverage_rebound_downtrend_router_v1,2322,2322,0.0,True,0.188386,0.207739,0.173488,703.795741,227.306769,20.986375,False,8.335772
7,state_conditioned_actor_selector_v1,2322,2322,0.0,True,0.268944,0.203634,0.314367,970.446343,18.644654,34.774907,False,11.638144


## Top Candidate vs Teacher Target Readout

,candidate,target_rows,matched_rows,missing_target_ratio,coverage_gate,position_tracking_mae,rebound_position_mae,risk_cap_violation,window_gate_loss,max_relative_gap,max_drawdown_gap,window_gate,alignment_loss
0,state_conditioned_rebound_risk_router_v1,2322,2322,0.0,True,0.0,0.151144,0.015691,134.348893,77.569598,2.939129,False,1.708542


In [4]:
selection = {
    'experiment': EXP,
    'decision': summary['decision'],
    'promotion': 'blocked_no_window_gate' if not summary.get('selected_window_gate') else 'review_required',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'reward'],
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'state_objective_training_target_v1',
    'teacher_candidate': target_selection['teacher_candidate'],
    'selected_candidate': summary['selected_candidate'],
    'target': target_selection['target'],
    'summary': summary,
    'artifacts': {
        'notebook': str((OUT_DIR / f'{EXP}.ipynb').relative_to(REPO_ROOT)),
        'alignment_scores': str(scores_path.relative_to(REPO_ROOT)),
        'summary': str(summary_path.relative_to(REPO_ROOT)),
        'selection': str((OUT_DIR / 'selection.json').relative_to(REPO_ROOT)),
    },
    'external_references': screen_selection.get('external_references', []),
    'conclusion': [
        'Existing candidates were scored against the per-date state-objective target table.',
        'The alignment winner is a diagnostic teacher/checkpoint-scoring input, not runtime promotion, unless it also clears the window gate and coverage gate.',
        'Next executable step is to feed target-alignment loss into checkpoint selection or behavior-cloning/PPO reward shaping and rerun walk-forward validation.',
    ],
}
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')

display(Markdown(
    f"""## Conclusion\n\n"
    f"- decision: `{selection['decision']}`\n"
    f"- selected candidate: `{selection['selected_candidate']}`\n"
    f"- teacher candidate: `{selection['teacher_candidate']}`\n"
    f"- alignment loss: `{summary['selected_alignment_loss']:.4f}`\n"
    f"- window gate: `{summary['selected_window_gate']}`; coverage gate: `{summary['selected_coverage_gate']}`\n\n"
    "This screen consumes the target table but still blocks promotion if the selected candidate does not clear the 30pp/15% walk-forward gate."""
))
selection


## Conclusion

"
    f"- decision: `target_alignment_no_promotion`
"
    f"- selected candidate: `state_conditioned_rebound_risk_router_v1`
"
    f"- teacher candidate: `state_conditioned_rebound_risk_router_v1`
"
    f"- alignment loss: `1.7085`
"
    f"- window gate: `False`; coverage gate: `True`

"
    "This screen consumes the target table but still blocks promotion if the selected candidate does not clear the 30pp/15% walk-forward gate.

{'experiment': 'state_objective_alignment_screen_v1',
 'decision': 'target_alignment_no_promotion',
 'promotion': 'blocked_no_window_gate',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'reward'],
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'state_objective_training_target_v1',
 'teacher_candidate': 'state_conditioned_rebound_risk_router_v1',
 'selected_candidate': 'state_conditioned_rebound_risk_router_v1',
 'target': {'relative_return_min_pp': 30.0, 'max_dd_max_pct': 15.0},
 'summary': {'decision': 'target_alignment_no_promotion',
  'selected_candidate': 'state_conditioned_rebound_risk_router_v1',
  'selected_alignment_loss': 1.7085418563344201,
  'selected_window_gate': False,
  'selected_coverage_gate': True,
  'selected_missing_target_ratio': 0.0,
  'next_mechanism': 'train_or_checkpoint_score_state_objective_targets',
  'required_terms': ['target_position_alignment_loss',
   'rebound_participation_alignment_loss',
  